# 09 時系列基盤モデル(Tier 4 — univariate forecasting)

Tier 4 は系列の **自己履歴から将来を予測** する別パラダイム(Tier0-3 の横断回帰とは別物)。`quantkit.models.foundation` は依存ゼロの **古典ベースライン**(seasonal-naive / AR)を常備し、重い基盤モデル(Chronos/TimesFM, torch 必須)は **import-gated アダプタ** で任意。基盤モデルは *この古典ベースラインを超えて* 初めて意味を持つ。

In [ ]:
import numpy as np
import pandas as pd

# Synthetic panel where forward returns are partly predictable (mild AR(1)
# momentum) so a model has signal to learn. No network.
rng = np.random.default_rng(3)
idx = pd.bdate_range('2015-01-01', periods=1600)
names = [f'A{i}' for i in range(8)]
n, k = len(idx), len(names)
r = rng.normal(0, 0.012, (n, k))
for t in range(1, n):
    r[t] += 0.06 * r[t - 1]
rets = pd.DataFrame(r, index=idx, columns=names)
prices = 100 * np.exp(rets.cumsum())
prices.iloc[-1].round(1)

## 1 系列を history / holdout に分け、古典予測器を当てる

In [ ]:
from quantkit import models as MD

series = prices['A0']
H = 21
history, actual = series.iloc[:-H], series.iloc[-H:]
forecasters = [MD.SeasonalNaiveForecaster(season=1), MD.SeasonalNaiveForecaster(season=5),
               MD.ARForecaster(p=5), MD.ARForecaster(p=20)]
rows = []
preds = {}
for fc in forecasters:
    p = fc.fit(history).predict(H)
    preds[fc.name] = p
    mae = float(np.mean(np.abs(p - actual.to_numpy())))
    rows.append({'forecaster': fc.name, 'MAE': mae})
pd.DataFrame(rows).set_index('forecaster').round(3)

## 予測 vs 実績

In [ ]:
import plotly.graph_objects as go
from quantkit.visualization.theme import style

fig = go.Figure()
fig.add_trace(go.Scatter(x=actual.index, y=actual.to_numpy(), name='actual', mode='lines+markers'))
for name, p in preds.items():
    fig.add_trace(go.Scatter(x=actual.index, y=p, name=name, mode='lines'))
style(fig, 'Tier-4 forecasters vs actual (holdout)').show()

## 基盤モデル(Chronos/TimesFM)は任意依存 — 無ければ明確に失敗

In [ ]:
try:
    MD.load_foundation('chronos')
except ImportError as e:
    print('foundation model unavailable (optional):', str(e)[:90], '...')

基盤モデルは torch + 重みを要するため quantkit の必須依存にはしない。常備の古典ベースラインが比較の土台で、基盤モデルはこれを超えて初めて採用に値する。

**正直な注記**: 合成データ。univariate 予測を横断バックテストへ接続するには、各資産の予測を信号化する追加ステップが要る(設計の余地として明示)。*方法論*の実演。